# 04 — Quantile models (GBQ + QR-LSTM)
Gradient-boosting quantile regression and a compact monotone-head QR-LSTM. Writes `model_metrics.json`, `model_preds.npz`.

In [ ]:
# === Colab/local auto-setup (device + data path) ===
import sys, os, subprocess
from pathlib import Path
def _pip(*pkgs):
    for p in pkgs:
        mod = p.split('==')[0].replace('-', '_').replace('scikit_learn', 'sklearn')
        try:
            __import__(mod)
        except Exception:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p])
_pip('numpy', 'pandas', 'scipy', 'scikit-learn', 'statsmodels', 'torch', 'matplotlib')
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
CSV = 'Bangladesh Meterological data.csv'
cands = [Path.cwd()/CSV, Path('/content')/CSV, Path(r'd:\BUET RESEARCH WORK\Bangladesh Flood')/CSV]
root = next((c.parent for c in cands if c.exists()), None)
if root is None:
    try:
        from google.colab import files
        files.upload(); root = Path.cwd()
    except Exception:
        raise FileNotFoundError('Upload "%s" next to this notebook.' % CSV)
os.environ['DFAA_ROOT'] = str(root)
print('DFAA_ROOT =', root)


In [ ]:
%%writefile common.py
"""Common config, data loader, and leak-free helpers for the Bangladesh DFAA study.

All paths hardcoded (workspace convention). Train-only fits everywhere.
Notation matches EXPERIMENT_DESIGN.md. No experiment numbers are produced here;
this is the shared, smoke-testable core that the notebooks reuse.
"""
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

# ROOT overridable so the shipped notebooks run on Colab/local (set env DFAA_ROOT).
ROOT = Path(os.environ.get("DFAA_ROOT", r"d:\BUET RESEARCH WORK\Bangladesh Flood"))
RAW_CSV = ROOT / "Bangladesh Meterological data.csv"
ART = ROOT / "artefacts"
ART.mkdir(exist_ok=True)

# ---- locked study constants (EXPERIMENT_DESIGN.md defaults) ----
SEED = 0
VARS_Z = ["Rainfall_mm", "Soil_moisture_mm"]          # standardized by train climatology
# robust standardization: per-(s,m) sd floored at SD_FLOOR_FRAC * station-pooled train sd,
# then z clipped to +-Z_CLIP. Guards the soil-moisture saturation / dry-month near-zero-sd
# pathology (otherwise z -> ~-40000). Fixed/train-only transforms => no leakage; train cells
# (|z|<3.6) are untouched. Documented in RESULTS_LOG S1/S2.
SD_FLOOR_FRAC = 0.15
Z_CLIP = 4.0
W_WEIGHTS = (1 / 3, 1 / 3, 1 / 3)                     # w1*z_P + w2*z_SM + w3*SPEI
ALPHA_DFAA = 1.8                                       # Wu (2006) constant in Eq. 2
LEADS = (1, 2, 3)                                      # symmetric window scale = lead h
TAUS = np.array([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95], dtype=np.float64)
THETA_PCT = 80.0                                       # theta_D = 80th pct of |DFAA| on train

# time-ordered split by ORIGIN year (the month t at which DFAA(s,t) is anchored)
TRAIN_YEARS = (2000, 2014)
VAL_YEARS = (2015, 2017)
TEST_YEARS = (2018, 2022)

# BMD station latitudes/longitudes (deg) for PET extraterrestrial radiation + maps.
# Standard BMD station coordinates; for PET only latitude matters (Ra is ~flat to +-0.2 deg).
# Provenance flagged for final verification before the .tex (CLAUDE.md Rule 3).
STATION_LATLON = {
    "Barisal":              (22.70, 90.37),
    "Bogra":                (24.85, 89.37),
    "Chittagong(Air-port)": (22.25, 91.81),
    "Comilla":              (23.43, 91.18),
    "Cox's Bazar":          (21.45, 91.97),
    "Dhaka":                (23.78, 90.38),
    "Faridpur":             (23.60, 89.85),
    "Jessore":              (23.18, 89.16),
    "Khulna":               (22.78, 89.53),
    "Mymensingh":           (24.75, 90.43),
    "Rajshahi":             (24.37, 88.70),
    "Rangpur":              (25.73, 89.23),
    "Sylhet":               (24.90, 91.88),
}


def load_clean():
    """Load raw CSV, drop the trailing all-NaN row, sort, add integer month index t.

    Returns a tidy long DataFrame with columns:
      Station_Name, Station_Code, Year, Month, Max_Temp, Min_Temp, Rainfall_mm,
      Soil_moisture_mm, SPEI_3, s (0..12 station id), t (0-based global month index),
      origin_year, split.
    Raw file is never modified.
    """
    df = pd.read_csv(RAW_CSV)
    # drop rows that are entirely NaN in the value columns (the trailing NaN row)
    val_cols = ["Max_Temp", "Min_Temp", "Rainfall_mm", "Soil_moisture_mm", "SPEI_3"]
    before = len(df)
    df = df.dropna(subset=["Station_Name", "Year", "Month"], how="any").copy()
    df = df.dropna(subset=val_cols, how="all").copy()
    dropped = before - len(df)

    df["Year"] = df["Year"].astype(int)
    df["Month"] = df["Month"].astype(int)
    df = df.sort_values(["Station_Name", "Year", "Month"]).reset_index(drop=True)

    stations = sorted(df["Station_Name"].unique())
    sid = {name: i for i, name in enumerate(stations)}
    df["s"] = df["Station_Name"].map(sid)

    # global 0-based month index over 2000-01 .. 2022-12
    df["t"] = (df["Year"] - 2000) * 12 + (df["Month"] - 1)

    def split_of(y):
        if TRAIN_YEARS[0] <= y <= TRAIN_YEARS[1]:
            return "train"
        if VAL_YEARS[0] <= y <= VAL_YEARS[1]:
            return "val"
        return "test"

    df["origin_year"] = df["Year"]
    df["split"] = df["Year"].map(split_of)
    return df, stations, sid, dropped


def to_grid(df, col):
    """Return an (S, T) float array of `col` indexed by [station s, month index t],
    NaN where missing. S=13 stations, T=276 months (2000-01..2022-12)."""
    S = df["s"].nunique()
    T = 276
    g = np.full((S, T), np.nan, dtype=np.float64)
    g[df["s"].to_numpy(), df["t"].to_numpy()] = df[col].to_numpy(dtype=float)
    return g


def train_mask_t(years=TRAIN_YEARS):
    """Boolean length-276 mask of month indices whose calendar year is in `years`."""
    t = np.arange(276)
    yr = 2000 + t // 12
    return (yr >= years[0]) & (yr <= years[1])


def fit_climatology(grid, train_t):
    """Per (station s, calendar month m) mean/std on TRAIN months only, with a robust
    std floor at SD_FLOOR_FRAC * station-pooled train sd (guards saturated/dry near-zero-sd
    cells). grid: (S,T); train_t: bool length T. Returns mu,sd as (S,12)."""
    S, T = grid.shape
    mu = np.full((S, 12), np.nan)
    sd = np.full((S, 12), np.nan)
    months = np.arange(T) % 12
    for s in range(S):
        for m in range(12):
            sel = (months == m) & train_t
            vals = grid[s, sel]
            vals = vals[~np.isnan(vals)]
            if len(vals) >= 2:
                mu[s, m] = vals.mean()
                sd[s, m] = vals.std(ddof=1)
    glob = np.nanstd(grid[:, train_t])
    for s in range(S):
        stat_sd = np.nanstd(grid[s, train_t])
        floor = SD_FLOOR_FRAC * stat_sd if np.isfinite(stat_sd) and stat_sd > 1e-6 else glob
        floor = max(floor, 1e-6)
        for m in range(12):
            if not np.isfinite(sd[s, m]) or sd[s, m] < floor:
                sd[s, m] = floor
            if not np.isfinite(mu[s, m]):
                mu[s, m] = np.nanmean(grid[s, train_t])
    return mu, sd


def standardize(grid, mu, sd):
    """z_v(s,t) = clip( (x - mu[s,m]) / sd[s,m], -Z_CLIP, +Z_CLIP ), m = t%12. NaNs propagate."""
    S, T = grid.shape
    months = np.arange(T) % 12
    z = (grid - mu[:, months]) / sd[:, months]
    return np.clip(z, -Z_CLIP, Z_CLIP)


In [ ]:
%%writefile evalkit.py
"""Shared probabilistic-forecast evaluation (reused by Steps 3-6).

All metrics take predictive quantiles Q[N, nT] (monotone non-decreasing in tau) and targets y[N].
CRPS uses the quantile estimator CRPS ~= 2*mean_tau(pinball_tau) (EXPERIMENT_DESIGN Step 7);
coarse but identical across methods, so CRPSS comparisons are fair.
"""
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

from common import TAUS


def pinball_per_tau(Q, y, taus=TAUS):
    err = y[:, None] - Q                       # [N,nT]
    pb = np.maximum(taus[None, :] * err, (taus[None, :] - 1) * err)
    return pb.mean(axis=0)                      # [nT]


def pinball(Q, y, taus=TAUS):
    return float(pinball_per_tau(Q, y, taus).mean())


def crps(Q, y, taus=TAUS):
    return float(2.0 * pinball_per_tau(Q, y, taus).mean())


def coverage(Q, y, lo_i, hi_i):
    lo, hi = Q[:, lo_i], Q[:, hi_i]
    inside = (y >= lo) & (y <= hi)
    return float(inside.mean()), float((hi - lo).mean())


def cdf_at(Q, thr, taus=TAUS):
    """Predictive CDF at threshold thr via linear interpolation across the quantile grid."""
    Q = np.asarray(Q, float)
    N, m = Q.shape
    thr_arr = np.full(N, thr) if np.isscalar(thr) else np.asarray(thr, float)
    k = np.sum(Q < thr_arr[:, None], axis=1)
    lo = np.clip(k - 1, 0, m - 1); hi = np.clip(k, 0, m - 1)
    ar = np.arange(N)
    qlo, qhi = Q[ar, lo], Q[ar, hi]
    tlo, thi = taus[lo], taus[hi]
    gap = qhi - qlo
    frac = np.where(gap > 1e-9, (thr_arr - qlo) / np.where(gap > 1e-9, gap, 1.0), 0.0)
    cdf = tlo + frac * (thi - tlo)
    cdf = np.where(k == 0, taus[0], cdf)
    cdf = np.where(k == m, taus[-1], cdf)
    return np.clip(cdf, 0.0, 1.0)


def event_metrics(Q, y, theta, taus=TAUS):
    """DTF (y>=+theta) and FTD (y<=-theta) detection from the predictive CDF."""
    out = {}
    p_dtf = 1.0 - cdf_at(Q, theta, taus)
    p_ftd = cdf_at(Q, -theta, taus)
    for name, p, ind in [("DTF", p_dtf, (y >= theta).astype(int)),
                          ("FTD", p_ftd, (y <= -theta).astype(int))]:
        d = {"base_rate": float(ind.mean()), "brier": float(brier_score_loss(ind, np.clip(p, 0, 1)))}
        if ind.sum() > 0 and ind.sum() < len(ind):
            d["auc"] = float(roc_auc_score(ind, p))
            d["pr_auc"] = float(average_precision_score(ind, p))
        else:
            d["auc"] = float("nan"); d["pr_auc"] = float("nan")
        out[name] = d
    return out


def all_metrics(Q, y, theta, crps_ref=None, taus=TAUS):
    """Bundle of probabilistic + calibration + event metrics for one method/lead/split."""
    pin = pinball(Q, y, taus)
    cr = crps(Q, y, taus)
    p80, w80 = coverage(Q, y, 1, 5)   # taus index 1=0.10, 5=0.90 -> 80% PI
    p90, w90 = coverage(Q, y, 0, 6)   # taus index 0=0.05, 6=0.95 -> 90% PI
    m = {"n": int(len(y)), "pinball": pin, "crps": cr,
         "picp80": p80, "width80": w80, "picp90": p90, "width90": w90}
    if crps_ref is not None and crps_ref > 0:
        m["crpss"] = float(1.0 - cr / crps_ref)
    m["event"] = event_metrics(Q, y, theta, taus)
    return m


def residual_quantiles(resid_train, taus=TAUS):
    """Empirical quantiles of training residuals -> additive spread for a point forecast."""
    r = resid_train[np.isfinite(resid_train)]
    return np.quantile(r, taus)


def point_to_quantiles(point, resid_q):
    """Q[N,nT] = point[:,None] + resid_q[None,:] (homoscedastic residual probabilization)."""
    Q = point[:, None] + resid_q[None, :]
    return np.maximum.accumulate(Q, axis=1)   # enforce monotone (sorted resid_q already monotone)


In [ ]:
"""Step 4 - quantile forecasting models for DFAA(s,t,h).

  * GBQ     : sklearn GradientBoostingRegressor(loss='quantile') per (lead, tau);
              predicted quantiles sorted to enforce non-crossing.
  * QR-LSTM : LSTM encoder over an L=12-month feature window + station embedding ->
              monotone non-crossing quantile head (cumsum of softplus increments, Cannon 2018),
              multi-lead output, masked pinball loss (Koenker-Bassett).

Same valid-DFAA origin set as the baselines (missing features mean-filled with TRAIN means).
Writes artefacts/model_metrics.json (+ raw test quantiles for Step 5 conformal) and appends RESULTS_LOG.
"""
import json
import time
import numpy as np
import torch
import torch.nn as nn
from sklearn.ensemble import GradientBoostingRegressor
from common import ART, load_clean, train_mask_t, LEADS, TAUS, SEED
import evalkit as ek

np.random.seed(SEED); torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("=" * 70); print(f"STEP 4 - quantile models (GBQ + QR-LSTM)  device={DEV}"); print("=" * 70)

df, stations, sid, _ = load_clean()
S, T = len(stations), 276
train_t = train_mask_t()
yr = 2000 + np.arange(T) // 12

fz = np.load(ART / "features.npz", allow_pickle=True)
feat = fz["feat"].astype(np.float64)            # (S,T,F)
fnames = list(fz["names"])
F = feat.shape[-1]
dz = np.load(ART / "dfaa.npz", allow_pickle=True)
DFAA = {h: dz[f"DFAA_h{h}"] for h in LEADS}
WE = {h: dz[f"WE_h{h}"] for h in LEADS}
meta = json.loads((ART / "dfaa_meta.json").read_text())
THETA = {h: meta["theta_D"][str(h)] for h in LEADS}

# ---- per-feature train mean/std (leak-free), for mean-fill + LSTM scaling ----
fmu = np.zeros(F); fsd = np.ones(F)
for j in range(F):
    v = feat[:, train_t, j]
    v = v[np.isfinite(v)]
    fmu[j] = v.mean(); fsd[j] = v.std() + 1e-8
feat_fill = np.where(np.isfinite(feat), feat, fmu[None, None, :])   # mean-filled (raw scale)
feat_z = (feat_fill - fmu[None, None, :]) / fsd[None, None, :]      # standardized (for LSTM)

we_mu = {h: float(np.nanmean(WE[h][:, train_t])) for h in LEADS}


def split_of(t):
    y = yr[t]
    return "train" if y <= 2014 else ("val" if y <= 2017 else "test")


# origins with valid target per lead
origins = {h: [(s, t) for s in range(S) for t in range(T) if np.isfinite(DFAA[h][s, t])] for h in LEADS}
spl = {h: {sp: [(s, t) for (s, t) in origins[h] if split_of(t) == sp] for sp in ["train", "val", "test"]}
       for h in LEADS}

# ================= GBQ =================
print("\n[GBQ] training GradientBoostingRegressor(loss='quantile') per (lead, tau)...")
t0 = time.time()
gbq_metrics = {}
gbq_Q = {h: {} for h in LEADS}   # store test/val raw quantiles for conformal
for h in LEADS:
    def Xy(idx):
        X = np.array([np.concatenate([feat_fill[s, t],
                      [WE[h][s, t] if np.isfinite(WE[h][s, t]) else we_mu[h]]]) for s, t in idx])
        y = np.array([DFAA[h][s, t] for s, t in idx])
        return X, y
    Xtr, ytr = Xy(spl[h]["train"])
    preds = {}
    for sp in ["val", "test"]:
        Xs, ys = Xy(spl[h][sp]); preds[sp] = (Xs, ys)
    Qcols = {sp: [] for sp in ["val", "test"]}
    for tau in TAUS:
        gb = GradientBoostingRegressor(loss="quantile", alpha=float(tau), n_estimators=300,
                                       max_depth=3, learning_rate=0.05, subsample=0.8,
                                       random_state=SEED)
        gb.fit(Xtr, ytr)
        for sp in ["val", "test"]:
            Qcols[sp].append(gb.predict(preds[sp][0]))
    gbq_metrics[h] = {}
    for sp in ["val", "test"]:
        Q = np.sort(np.stack(Qcols[sp], axis=1), axis=1)   # enforce non-crossing
        y = preds[sp][1]
        ref = ek.crps(np.tile(np.quantile(ytr, TAUS), (len(y), 1)), y)
        gbq_metrics[h][sp] = ek.all_metrics(Q, y, THETA[h], crps_ref=ref)
        gbq_Q[h][sp] = (Q, y)
print(f"  GBQ done in {time.time()-t0:.0f}s")

# ================= QR-LSTM =================
L = 12
nT = len(TAUS)
nL = len(LEADS)
ystd = {h: float(np.std([DFAA[h][s, t] for s, t in spl[h]["train"]])) for h in LEADS}  # target scale


def make_window_set(split):
    """Common origins (valid for ALL leads) in a split -> (Xwin, sid, Y[nL], M[nL])."""
    valid = [(s, t) for s in range(S) for t in range(T)
             if split_of(t) == split and t - L + 1 >= 0
             and all(np.isfinite(DFAA[h][s, t]) for h in LEADS)]
    Xw = np.array([feat_z[s, t - L + 1:t + 1] for s, t in valid], dtype=np.float32)  # (N,L,F)
    si = np.array([s for s, t in valid], dtype=np.int64)
    Y = np.array([[DFAA[h][s, t] / ystd[h] for h in LEADS] for s, t in valid], dtype=np.float32)
    idx = np.array(valid)
    return Xw, si, Y, idx


Xtr, sitr, Ytr, _ = make_window_set("train")
Xva, siva, Yva, idxva = make_window_set("val")
Xte, site, Yte, idxte = make_window_set("test")
print(f"\n[QR-LSTM] window L={L}  train {len(Xtr)} / val {len(Xva)} / test {len(Xte)} origins")


class QRLSTM(nn.Module):
    def __init__(self, F, nstat, hidden=48, emb=6):
        super().__init__()
        self.emb = nn.Embedding(nstat, emb)
        self.lstm = nn.LSTM(F, hidden, batch_first=True)
        trunk = hidden + emb
        self.base = nn.Linear(trunk, nL)
        self.inc = nn.Linear(trunk, nL * (nT - 1))

    def forward(self, x, si):
        out, (hn, cn) = self.lstm(x)
        z = torch.cat([hn[-1], self.emb(si)], dim=1)
        base = self.base(z)                                  # [B,nL]
        inc = torch.nn.functional.softplus(self.inc(z)).view(-1, nL, nT - 1)
        q = torch.cat([base.unsqueeze(-1), base.unsqueeze(-1) + torch.cumsum(inc, -1)], -1)
        return q                                              # [B,nL,nT]


def pinball_loss(q, y, taus):
    err = y.unsqueeze(-1) - q
    l = torch.maximum(taus.view(1, 1, nT) * err, (taus.view(1, 1, nT) - 1) * err)
    return l.mean()


model = QRLSTM(F, S).to(DEV)
opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=1e-4)
taus_t = torch.tensor(TAUS, dtype=torch.float32, device=DEV)
Xtr_t = torch.tensor(Xtr, device=DEV); sitr_t = torch.tensor(sitr, device=DEV); Ytr_t = torch.tensor(Ytr, device=DEV)
Xva_t = torch.tensor(Xva, device=DEV); siva_t = torch.tensor(siva, device=DEV)
EPOCHS = 250; BATCH = 128
best_val = np.inf; best_state = None
n = len(Xtr_t)
for ep in range(EPOCHS):
    model.train(); perm = torch.randperm(n)
    for i in range(0, n, BATCH):
        b = perm[i:i + BATCH]
        opt.zero_grad()
        q = model(Xtr_t[b], sitr_t[b])
        loss = pinball_loss(q, Ytr_t[b], taus_t)
        loss.backward(); opt.step()
    if (ep + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            qv = model(Xva_t, siva_t)
            vl = pinball_loss(qv, torch.tensor(Yva, device=DEV), taus_t).item()
        if vl < best_val:
            best_val = vl; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
print(f"  QR-LSTM best val pinball (scaled) = {best_val:.4f}")

# evaluate QR-LSTM per lead (unscale)
model.eval()
lstm_metrics = {}; lstm_Q = {h: {} for h in LEADS}
with torch.no_grad():
    for sp, Xs, sis, idxs in [("val", Xva, siva, idxva), ("test", Xte, site, idxte)]:
        q = model(torch.tensor(Xs, device=DEV), torch.tensor(sis, device=DEV)).cpu().numpy()  # [N,nL,nT]
        for hi, h in enumerate(LEADS):
            Q = q[:, hi, :] * ystd[h]
            y = np.array([DFAA[h][s, t] for s, t in idxs])
            ytr_h = np.array([DFAA[h][s, t] for s, t in spl[h]["train"]])
            ref = ek.crps(np.tile(np.quantile(ytr_h, TAUS), (len(y), 1)), y)
            lstm_metrics.setdefault(h, {})[sp] = ek.all_metrics(Q, y, THETA[h], crps_ref=ref)
            lstm_Q[h][sp] = (Q, y, idxs)

# save raw test/val quantiles for the conformal step (Step 5)
np.savez_compressed(ART / "model_preds.npz",
                    **{f"gbq_{h}_{sp}_Q": gbq_Q[h][sp][0] for h in LEADS for sp in ["val", "test"]},
                    **{f"gbq_{h}_{sp}_y": gbq_Q[h][sp][1] for h in LEADS for sp in ["val", "test"]},
                    **{f"lstm_{h}_{sp}_Q": lstm_Q[h][sp][0] for h in LEADS for sp in ["val", "test"]},
                    **{f"lstm_{h}_{sp}_y": lstm_Q[h][sp][1] for h in LEADS for sp in ["val", "test"]},
                    **{f"lstm_{h}_{sp}_idx": lstm_Q[h][sp][2] for h in LEADS for sp in ["val", "test"]})

out = {"gbq": gbq_metrics, "lstm": lstm_metrics}
(ART / "model_metrics.json").write_text(json.dumps(out, indent=2, default=float))

print("\n" + "=" * 70); print("MODEL SUMMARY (TEST) - CRPS / CRPSS / PICP90 / DTF-AUC / FTD-AUC"); print("=" * 70)
for h in LEADS:
    print(f"\n-- lead h={h} --")
    print(f"{'model':10s} {'CRPS':>7s} {'CRPSS':>7s} {'PICP80':>7s} {'PICP90':>7s} {'DTF-AUC':>8s} {'FTD-AUC':>8s}")
    for name, mm in [("GBQ", gbq_metrics), ("QR-LSTM", lstm_metrics)]:
        m = mm[h]["test"]; ev = m["event"]
        print(f"{name:10s} {m['crps']:7.4f} {m.get('crpss', float('nan')):7.3f} {m['picp80']:7.3f} "
              f"{m['picp90']:7.3f} {ev['DTF']['auc']:8.3f} {ev['FTD']['auc']:8.3f}")
print(f"\nWrote {ART/'model_metrics.json'} and {ART/'model_preds.npz'}")
print("\nSTEP 4 OK.")
